In [1]:
# 07_recourse.ipynb
# Cross-Domain AI Governance Gap Analysis
# 07. Recourse Feasibility Axis
#
# Add a recourse-feasibility axis to the GGI (oriented so higher = better):
#   s_REC = (# adversely-predicted individuals with a FEASIBLE counterfactual)
#           / (# adversely-predicted individuals)
# "Adverse" = predicted high-risk (label 1). A counterfactual is feasible if it flips the
# decision by changing MUTABLE features only (immutable attributes frozen). Counterfactuals
# are generated with DiCE on the raw (pre-encoding) feature space. Recourse is computed only
# for datasets with interpretable features; Porto Seguro uses anonymized proxies, so recourse
# is undefined there and reported as N/A. The recourse axis is layered onto the
# noise-corrected component scores from notebook 04.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import dice_ml
from dice_ml import Dice

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
N_ADVERSE = 100
N_CF = 3

def load_raw(key):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}.parquet"))

def spec_german():
    df = load_raw("german").drop(columns=["personal_status_sex", "sex", "age_group"], errors="ignore")
    immutable = ["age", "foreign_worker"]
    continuous = [c for c in ["duration", "credit_amount", "age", "installment_rate",
                              "residence_since", "existing_credits", "num_dependents"] if c in df.columns]
    return df, "target", immutable, continuous

def spec_credit_card():
    df = load_raw("credit_card_approval").drop(
        columns=["ID", "sex", "target_30dpd", "target_60dpd", "domain"], errors="ignore")
    immutable = [c for c in ["CODE_GENDER", "DAYS_BIRTH", "FLAG_OWN_REALTY", "FLAG_OWN_CAR"] if c in df.columns]
    continuous = [c for c in ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "DAYS_BIRTH",
                              "DAYS_EMPLOYED", "CNT_FAM_MEMBERS"] if c in df.columns]
    return df, "target", immutable, continuous

def spec_medical():
    df = load_raw("insurance").drop(columns=["age_group", "smoker_flag", "domain", "charges"], errors="ignore")
    immutable = ["age", "sex"]
    continuous = [c for c in ["age", "bmi", "children"] if c in df.columns]
    return df, "target", immutable, continuous

SPECS = {"German Credit": spec_german, "Credit Card Approval": spec_credit_card, "Medical Cost": spec_medical}
UNDEFINED = ["Porto Seguro"]

def compute_srec(df, target, immutable, continuous, n_adverse=N_ADVERSE, n_cf=N_CF, seed=RANDOM_STATE):
    y = df[target].astype(int)
    X = df.drop(columns=[target])
    cont = [c for c in continuous if c in X.columns]
    cat = [c for c in X.columns if c not in cont]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y)
    pre = ColumnTransformer([("num", StandardScaler(), cont),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    clf = Pipeline([("pre", pre),
                    ("model", RandomForestClassifier(n_estimators=300, max_depth=8,
                        class_weight="balanced", random_state=seed, n_jobs=-1))])
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    adverse_idx = X_test.index[pred == 1][:n_adverse]
    if len(adverse_idx) == 0:
        return np.nan, 0, 0
    query = X_test.loc[adverse_idx].copy()
    features_to_vary = [c for c in X.columns if c not in immutable]
    train_df = X_train.copy(); train_df[target] = y_train.values
    d = dice_ml.Data(dataframe=train_df, continuous_features=cont, outcome_name=target)
    m = dice_ml.Model(model=clf, backend="sklearn")
    exp = Dice(d, m, method="random")
    feasible, tested = 0, 0
    for i in range(len(query)):
        tested += 1
        try:
            cf = exp.generate_counterfactuals(query.iloc[[i]], total_CFs=n_cf, desired_class=0,
                                              features_to_vary=features_to_vary)
            cf_df = cf.cf_examples_list[0].final_cfs_df
            if cf_df is not None and len(cf_df) > 0:
                feasible += 1
        except Exception:
            pass
    return (feasible / tested if tested else np.nan), feasible, tested

rec_rows = []
for name, spec_fn in SPECS.items():
    df, target, immutable, continuous = spec_fn()
    s_rec, feasible, tested = compute_srec(df, target, immutable, continuous)
    rec_rows.append({"dataset": name, "s_REC": round(s_rec, 4) if np.isfinite(s_rec) else np.nan,
                     "feasible": feasible, "tested": tested, "immutable": ", ".join(immutable)})
    print(f"[{name:>22}] s_REC={s_rec:.3f}  ({feasible}/{tested} adverse cases)")
for name in UNDEFINED:
    rec_rows.append({"dataset": name, "s_REC": np.nan, "feasible": 0, "tested": 0,
                     "immutable": "N/A (anonymized proxies)"})
    print(f"[{name:>22}] s_REC=N/A (recourse undefined for anonymized features)")
rec_df = pd.DataFrame(rec_rows)
print(rec_df.to_string(index=False))
rec_df.to_csv(os.path.join(TAB_DIR, "07_recourse_scores.csv"), index=False)

# Layer recourse onto the noise-corrected component scores from notebook 04
REC_THRESHOLDS = {"KR_AI_Basic_Act": 0.60, "EU_AI_Act": 0.70, "NAIC": 0.65}
BASE_REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}
comp_scores = pd.read_csv(os.path.join(TAB_DIR, "04_component_scores.csv")).merge(
    rec_df[["dataset", "s_REC"]], on="dataset", how="left")

def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

ext_rows = []
for _, r in comp_scores.iterrows():
    for reg, tau in BASE_REGULATIONS.items():
        gaps = [component_gap(r["s_PERF"], tau["s_PERF"]), component_gap(r["s_DI"], tau["s_DI"]),
                component_gap(r["s_SP"], tau["s_SP"]), component_gap(r["s_EXPL"], tau["s_EXPL"])]
        g_rec = component_gap(r["s_REC"], REC_THRESHOLDS[reg]) if np.isfinite(r["s_REC"]) else np.nan
        if np.isfinite(g_rec):
            gaps.append(g_rec)
        valid = [g for g in gaps if np.isfinite(g)]
        ggi_ext = float(np.sqrt(np.sum(np.square(valid)))) if valid else np.nan
        ext_rows.append({"domain": r["domain"], "dataset": r["dataset"], "regulation": reg,
                         "s_REC": r["s_REC"], "g_REC": g_rec, "GGI_extended": round(ggi_ext, 4),
                         "recourse_included": bool(np.isfinite(g_rec))})
ext_df = pd.DataFrame(ext_rows)
print(ext_df.to_string(index=False))
ext_df.to_csv(os.path.join(TAB_DIR, "07_ggi_extended.csv"), index=False)

base = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"))
base_kr = base[base["regulation"] == "KR_AI_Basic_Act"][["dataset", "GGI"]].rename(columns={"GGI": "GGI_4axis"})
ext_kr = ext_df[ext_df["regulation"] == "KR_AI_Basic_Act"][["dataset", "GGI_extended", "recourse_included"]]
compare = base_kr.merge(ext_kr, on="dataset")
print(compare.to_string(index=False))
compare.to_csv(os.path.join(TAB_DIR, "07_ggi_recourse_comparison.csv"), index=False)
print("07_recourse complete")


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

...

100%|██████████| 1/1 [00:00<00:00,  1.19it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

[          Medical Cost] s_REC=1.000  (63/63 adverse cases)
[          Porto Seguro] s_REC=N/A (recourse undefined for anonymized features)
             dataset  s_REC  feasible  tested                                              immutable
       German Credit   1.00        98      98                                    age, foreign_worker
Credit Card Approval   0.88        88     100 CODE_GENDER, DAYS_BIRTH, FLAG_OWN_REALTY, FLAG_OWN_CAR
        Medical Cost   1.00        63      63                                               age, sex
        Porto Seguro    NaN         0       0                               N/A (anonymized proxies)
   domain              dataset      regulation  s_REC  g_REC  GGI_extended  recourse_included
   credit        German Credit KR_AI_Basic_Act   1.00    0.0        0.6874               True
   credit        German Credit       EU_AI_Act   1.00    0.0        0.7395               True
   credit        German Credit            NAIC   1.00    0.0        0.715